# Imports

In [ ]:
!pip install torch_geometric -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 920.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.5 MB/s eta 0:00:00


In [ ]:
import os
import torch
import numpy as np
from tqdm import tqdm
from torch_geometric.data import Data

In [ ]:
# Change directory
os.chdir('/content/drive/MyDrive/nids-mitre/')

!pwd


/content/drive/MyDrive/nids-mitre


# Functions

In [ ]:
def calc_shannon_entropy(labels, num_categories=7):
    if len(labels) == 0: return 0.0
    # Count the frequency of each port category (0-6)
    counts = torch.bincount(labels, minlength=num_categories).float()
    probs = counts / counts.sum()
    probs = probs[probs > 0] # Avoid log(0)
    return -torch.sum(probs * torch.log2(probs)).item()

In [ ]:
def generate_v2_dataset(src_path, dst_path, splits, max_entropy):
    # Destination directories
    if not os.path.exists(dst_path):
        os.makedirs(dst_path)
        for split in splits:
            os.makedirs(os.path.join(dst_path, split))

    # Process each split
    for split in splits:
        src_split_path = os.path.join(src_path, split)
        dst_split_path = os.path.join(dst_path, split)

        files = [f for f in os.listdir(src_split_path) if f.endswith('.pt')]
        print(f"\nProcessing split: {split} ({len(files)} files)")

        for filename in tqdm(files):
            # Load original
            data = torch.load(os.path.join(src_split_path, filename), weights_only=False)

            # --- ENTROPY CALCULATION ---
            src_idx, dst_idx = data.edge_index
            num_nodes = data.x.size(0)

            # Collapse one-hot ports (assuming columns 0 to 7)
            port_labels = torch.argmax(data.edge_attr[:, 0:7], dim=1)

            out_entropy = torch.zeros(num_nodes, 1)
            in_entropy = torch.zeros(num_nodes, 1)

            # Only iterate over nodes that have connections
            unique_src = src_idx.unique()
            for n in unique_src:
                out_entropy[n] = calc_shannon_entropy(port_labels[src_idx == n])

            unique_dst = dst_idx.unique()
            for n in unique_dst:
                in_entropy[n] = calc_shannon_entropy(port_labels[dst_idx == n])

            # Normalization and packaging
            # node_stats becomes [num_nodes, 2]
            data.node_stats = torch.cat([out_entropy, in_entropy], dim=1) / max_entropy

            # Save to new location
            torch.save(data, os.path.join(dst_split_path, filename))

# Main

In [ ]:
SRC_ROOT = "./dataset_processed"
DST_ROOT = "./dataset_processed_entropy"
SPLITS = ['train', 'val', 'test']
MAX_ENTROPY = np.log2(7) # Theoretical maximum for 7 categories


generate_v2_dataset(SRC_ROOT, DST_ROOT, SPLITS, MAX_ENTROPY)


Processing split: train (1998 files)


100%|██████████| 1998/1998 [02:54<00:00, 11.47it/s]



Processing split: val (428 files)


100%|██████████| 428/428 [01:16<00:00,  5.57it/s]



Processing split: test (429 files)


100%|██████████| 429/429 [00:36<00:00, 11.83it/s]


In [ ]:
SRC_ROOT = "./dataset_processed_thu0103"
DST_ROOT = "./dataset_processed_thu0103_entropy"
SPLITS = ['test2']
MAX_ENTROPY = np.log2(7) # Theoretical maximum for 7 categories


generate_v2_dataset(SRC_ROOT, DST_ROOT, SPLITS, MAX_ENTROPY)


Processing split: test2 (2822 files)


100%|██████████| 2822/2822 [03:37<00:00, 12.98it/s]
